# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/helnagar123/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
!pip -q install duckdb datasets huggingface_hub pyarrow

In [4]:
import duckdb

from datasets import load_dataset
from google.colab import userdata
from huggingface_hub import login

In [5]:
HF_TOKEN = userdata.get("HF_TOKEN")

login(token=HF_TOKEN)

print("Login successful")

Login successful


In [6]:
con = duckdb.connect()

print("DuckDB Ready")

DuckDB Ready


In [7]:
content = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    split="train"
)

query90 = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_query_90d",
    split="train"
)

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/519606 [00:00<?, ? examples/s]

fact_content_query_90d.parquet: reconstructing file:   0%|          |  0.00B / 60.7MB            

fact_content_query_90d.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2414248 [00:00<?, ? examples/s]

In [8]:
content_arrow = content.data.table
query_arrow = query90.data.table

con.register("dim_content", content_arrow)
con.register("fact_query90", query_arrow)

print("Ready")

Ready


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [9]:
feature_vector = con.sql("""
SELECT
    dc.content_hash_id,
    fq.query_hash_id,

    dc.search_volume,
    dc.competition,
    dc.word_count,
    dc.backlinks,
    fq.avg_position_90d

FROM dim_content dc
JOIN fact_query90 fq
USING(client_hash_id, content_hash_id)

LIMIT 10000
""").df()

feature_vector.head()

,content_hash_id,query_hash_id,search_volume,competition,word_count,backlinks,avg_position_90d
0,content_106c427eb8e5cf7c,query_f107ac4d73e49e19,0,0.0,2235,0,5.320988
1,content_10b10a9b5cdf569f,query_dcb219c21af35635,0,0.0,2263,0,0.000000
2,content_10c55a1a67ee4e33,query_ee044ca4ed207661,40,0.0,2399,23,8.153846
3,content_11b345dfc3b7f221,query_e23edc2ddc1c2a7e,0,0.0,2413,0,25.100000
4,content_1212c2fcd4ca678c,query_ffd3a0332c4f1933,10,0.0,2346,0,5.416667


## 2. Feature notes (meaning, missing, categorical, available-when?)

search_volume
- Meaning: Monthly keyword demand.
- Missing: Fill with 0.
- Type: Numeric.
- Available when: Before the refresh decision.

competition
- Meaning: Keyword competition.
- Missing: Fill with median.
- Type: Numeric.
- Available when: Before prediction.

word_count
- Meaning: Current content length.
- Missing: Fill with median.
- Type: Numeric.
- Available when: Existing content.

backlinks
- Meaning: Existing authority.
- Missing: Fill with 0.
- Type: Numeric.
- Available when: Historical.

avg_position_90d
- Meaning: Average ranking.
- Missing: Remove missing rows.
- Type: Numeric.
- Available when: Historical.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [10]:
leak_df = feature_vector.copy()

leak_df["leaky_feature"] = con.sql("""
SELECT clicks_90d
FROM fact_query90
LIMIT 10000
""").df()["clicks_90d"]

leak_df.head()

,content_hash_id,query_hash_id,search_volume,competition,word_count,backlinks,avg_position_90d,leaky_feature
0,content_106c427eb8e5cf7c,query_f107ac4d73e49e19,0,0.0,2235,0,5.320988,0
1,content_10b10a9b5cdf569f,query_dcb219c21af35635,0,0.0,2263,0,0.000000,0
2,content_10c55a1a67ee4e33,query_ee044ca4ed207661,40,0.0,2399,23,8.153846,0
3,content_11b345dfc3b7f221,query_e23edc2ddc1c2a7e,0,0.0,2413,0,25.100000,0
4,content_1212c2fcd4ca678c,query_ffd3a0332c4f1933,10,0.0,2346,0,5.416667,0


In [11]:
leak_df = leak_df.drop(columns=["leaky_feature"])
print("Leaky feature removed.")

Leaky feature removed.


## 4. What I excluded and why

client_hash_id
Excluded because it identifies the client and is not predictive.

content_hash_id
Excluded because it is only an identifier.

query_hash_id
Excluded because it is only an identifier.

window_start
Excluded because it defines the reporting window rather than the content characteristics.

window_end
Excluded because it represents the observation period, not a feature available for prediction.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.